In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import transformers
print(f"Transformers: {transformers.__version__}")

Transformers: 5.0.0


In [4]:
from transformers import AutoModel, AutoTokenizer
import torch


model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model =AutoModel.from_pretrained(model_name)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Model on: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [5]:
text = "Hello, how are you doing today?"
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

text2 = "I love transformers and unbelievable NLP applications"
tokens2 = tokenizer.tokenize(text2)
print(f"Subwords: {tokens2}")

Tokens: ['hello', ',', 'how', 'are', 'you', 'doing', 'today', '?']
Subwords: ['i', 'love', 'transformers', 'and', 'unbelievable', 'nl', '##p', 'applications']


In [6]:
encoded = tokenizer(text, return_tensors="pt", padding=True, max_length=128)
print(f"input_ids shape: {encoded['input_ids'].shape}")
print(f"attention_mask shape: {encoded['attention_mask'].shape}")
print(f"token_type_ids shape: {encoded['token_type_ids'].shape}")

input_ids shape: torch.Size([1, 10])
attention_mask shape: torch.Size([1, 10])
token_type_ids shape: torch.Size([1, 10])


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [7]:
print(f"Decoded: {tokenizer.decode(encoder['input_ids'][0])}")

print(f"[CLS] id: {tokenizer.cls_token_id}")
print(f"[SEP] id: {tokenizer.sep_token_id}")
print(f"[MASK] id: {tokenizer.mask_token_id}")
print(f"[PAD] id: {tokenizer.pad_token_id}")
print(f"Vocab size: {tokenizer.vocab_size}")

Decoded: [CLS] hello, how are you doing today? [SEP]
[CLS] id: 101
[SEP] id: 102
[MASK] id: 103
[PAD] id: 0
Vocab size: 30522


In [9]:
texts = [
    "I love machine learning",
    "Natural language processing is fascinating and incredibly powerful"

]
encoded = tokenizer(texts, return_tensors='pt', padding=True, max_length=128)

print(f"Batch input shape: {encoded['input_ids'].shape}")
print(f"Attention mask:\n{encoded['attention_mask']}")

for i, text in enumerate(texts):
    ids = encoded['input_ids'][i]
    mask = encoded['attention_mask'][i]
    real_len = mask.sum().item()
    print(f"Text {i}: '{text}' -> {real_len} tokens (padded to {len(ids)})")

Batch input shape: torch.Size([2, 10])
Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Text 0: 'I love machine learning' -> 6 tokens (padded to 10)
Text 1: 'Natural language processing is fascinating and incredibly powerful' -> 10 tokens (padded to 10)


In [11]:
print(f"sentence 0 ids:\n {encoded['input_ids'][0]}")
print(f"sentence 0 Attention Mask:\n {encoded['attention_mask'][0]}")

sentence 0 ids:
 tensor([ 101, 1045, 2293, 3698, 4083,  102,    0,    0,    0,    0])
sentence 0 Attention Mask:
 tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])


In [16]:
text = "Machine learning is tranforming the world"
encoded = tokenizer(text, return_tensors="pt", padding=True, truncated=True)
encoded = {k: v.to(device) for k, v in encoded.items()}

In [28]:
print(encoded.keys(), sep=',')

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


In [18]:
tokens = tokenizer.tokenize(text)
print(f"Subwords: {tokens}")

Subwords: ['machine', 'learning', 'is', 'tran', '##form', '##ing', 'the', 'world']


In [19]:
encoded

{'input_ids': tensor([[  101,  3698,  4083,  2003, 25283, 14192,  2075,  1996,  2088,   102]],
        device='cuda:0'),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [22]:
with torch.no_grad():
    outputs = model(**encoded)

print(f"last_hidden_state shape: {outputs.last_hidden_state.shape}")

last_hidden_state shape: torch.Size([1, 10, 768])


In [23]:
for k, v in outputs.items():
    print(k, v)

last_hidden_state tensor([[[-0.0576,  0.2041,  0.0295,  ..., -0.4943,  0.1997,  0.5105],
         [ 0.2572,  0.3249,  0.0532,  ..., -0.2671,  0.5303,  0.5041],
         [-0.1712,  0.2949,  0.1548,  ..., -1.2496,  0.1078,  0.5392],
         ...,
         [-0.1130, -0.0105,  0.5316,  ..., -0.1340,  0.1603,  0.0855],
         [ 0.1832, -0.3262, -0.1863,  ...,  0.0407,  0.4515, -0.2007],
         [ 0.7156,  0.1664, -0.2900,  ..., -0.2423, -0.8107, -0.0718]]],
       device='cuda:0')
pooler_output tensor([[-0.9374, -0.4532, -0.5300,  0.8098,  0.2393, -0.4556,  0.8746,  0.4787,
         -0.4263, -1.0000, -0.2379,  0.8336,  0.9928,  0.0912,  0.9300, -0.7083,
         -0.5200, -0.6056,  0.4155, -0.5696,  0.7458,  0.9997,  0.2155,  0.4709,
          0.4665,  0.9534, -0.7779,  0.9604,  0.9617,  0.7712, -0.7885,  0.4522,
         -0.9950, -0.2336, -0.5965, -0.9905,  0.5090, -0.6687,  0.0508, -0.0974,
         -0.8925,  0.3496,  0.9998, -0.4897,  0.5258, -0.3672, -1.0000,  0.3431,
         -0.8910

In [34]:
# === 三种常用 pooling的方法 ===

# 方法一:[CLS] token(BERT官方的方法)
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f"[CLS] embedding shape: {cls_embedding.shape}\n")

# 方法二: Mean pooling(效果更加好， Sentence-BERT用这个)
attention_mask = encoded['attention_mask'].unsqueeze(-1) # shape (1, seq_len, 1)
sum_embeddings = (outputs.last_hidden_state * attention_mask).sum(dim=1)
count = attention_mask.sum(dim=1)
print(f"count shape: {count.shape}")
mean_embedding = sum_embeddings / count
print(f"Mean embedding shape: {mean_embedding.shape}\n")

# 方法三: Max Pooling
masked = outputs.last_hidden_state.clone()
masked[encoded['attention_mask'] == 0] = -1e9
max_embedding = masked.max(dim=1).values
print(f"Max embedding shape: {max_embedding.shape}")

[CLS] embedding shape: torch.Size([1, 768])

count shape: torch.Size([1, 1])
Mean embedding shape: torch.Size([1, 768])

Max embedding shape: torch.Size([1, 768])


In [33]:
print(f"{encoded['attention_mask'].shape}")

torch.Size([1, 10])


In [32]:
print(f"{outputs.last_hidden_state.shape}")

torch.Size([1, 10, 768])


In [39]:
from torch.nn.functional import cosine_similarity

def get_embedding(text, pooling='mean'):
    encoded = tokenizer(text, return_tensors="pt", padding=True, max_length=128)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    if pooling == 'cls':
        return outputs.last_hidden_state[:, 0, :]
    elif pooling == 'mean':
        masked = encoded['attention_mask'].unsqueeze(-1).float()
        return (outputs.last_hidden_state * masked).sum(dim=1) / masked.sum(dim=1)

sentences = [
    "I love programming in Python",
    "Python is my favorite coding language",
    "The weather is beautiful today",
    "I enjoy writing code",
    "It is sunny outside",
]

embeddings = torch.cat([get_embedding(s) for s in sentences])
print(f"All embedding shape: {embeddings.shape}")

print("\n=== Cosine Similarity Matrix ===")
print(f"{'':>40}", end="")
for i in range(len(sentences)):
    print(f"  S{i}", end="")
print()

for i in range(len(sentences)):
    print(f"S{i}: {sentences[i]:>38}", end="")
    for j in range(len(sentences)):
        sim = cosine_similarity(embeddings[i:i+1], embeddings[j:j+1]).item()
        print(f"  {sim:2f}", end="")
    print()

All embedding shape: torch.Size([5, 768])

=== Cosine Similarity Matrix ===
                                          S0  S1  S2  S3  S4
S0:           I love programming in Python  1.000000  0.815052  0.575439  0.813504  0.570773
S1:  Python is my favorite coding language  0.815052  1.000000  0.592862  0.732974  0.567650
S2:         The weather is beautiful today  0.575439  0.592862  1.000000  0.588965  0.811073
S3:                   I enjoy writing code  0.813504  0.732974  0.588965  1.000000  0.650705
S4:                    It is sunny outside  0.570773  0.567650  0.811073  0.650705  1.000000


In [41]:
from torch import nn
class SimpleBERTClassifier(nn.Module):
    def __init__(self, num_classes = 2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.classifier = nn.Linear(768, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

classifier = SimpleBERTClassifier(num_classes=2).to(device)
print(f"Total parameters: {sum(p.numel() for p in classifier.parameters()):,}")

dummy = tokenizer("This is a test", return_tensors="pt").to(device)
with torch.no_grad():
    logits = classifier(**dummy)
print(f"Logits shape: {logits.shape}")
print(f"Logits: {logits}")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters: 109,483,778
Logits shape: torch.Size([1, 2])
Logits: tensor([[0.1234, 0.2019]], device='cuda:0')
